In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df = pd.read_csv('../data/cancer_data.tsv',sep='\t')

In [6]:
df.head()

,RT @JarrodTheLord: Yall creating life and cloning animals but want us to believe there is no cure for Cancer or Aids... alright https://t.c…,no
0,RT @TommySobiesski: you could give GameStop th...,no
1,This last year has been the hardest of my life...,no
2,The year is 2043. You opened your newspaper an...,no
3,RT @gorskon: Here we go again. Yet another dub...,no
4,So sick of festival kids fingering themselves ...,no


In [7]:
df.shape

(1506, 2)

In [8]:
df.columns = ['reviews','cat']

In [9]:
df.head()

,reviews,cat
0,RT @TommySobiesski: you could give GameStop th...,no
1,This last year has been the hardest of my life...,no
2,The year is 2043. You opened your newspaper an...,no
3,RT @gorskon: Here we go again. Yet another dub...,no
4,So sick of festival kids fingering themselves ...,no


In [10]:
df['cat'].unique()

array(['no', 'yes', 'yes ', 'no ', ' no'], dtype=object)

In [11]:
mind = {'no':'no', 'no ': 'no', ' no':'no', 'yes':'yes', 'yes ':'yes'}

In [12]:
df['cat'] = [mind[x] for x in df['cat']]

In [13]:
df['cat'].unique()

array(['no', 'yes'], dtype=object)

In [15]:
pd.set_option('display.max_colwidth',200)
df.head()

,reviews,cat
0,RT @TommySobiesski: you could give GameStop the cure to cancer &amp; they’d offer you $3.89 https://t.co/REjIez4G7D,no
1,"This last year has been the hardest of my life - to all the cancer researchers, doctors, nurses and caregivers who are so selflessly dedicated to finding a cure and healing patients, you are my he...",no
2,The year is 2043. You opened your newspaper and read the headlines- - World Health Organization releases a cure for Cancer. - Global warming threat- eliminated! - Enrile celebrating his birthday...,no
3,RT @gorskon: Here we go again. Yet another dubious cancer cure video. Watch to the end and see! https://t.co/JObs8QeYiS,no
4,So sick of festival kids fingering themselves with amethyst and wearing 10 pashimas thinking they're hippies. Real hippies would get together and cure cancer instead of spending $600 to stand arou...,no


In [16]:
df['cat'].value_counts()

cat
no     1298
yes     208
Name: count, dtype: int64

In [18]:
import nltk
import re
import string

In [19]:
def remove_links(tweet):
    tweet = re.sub(r'http\S+','',tweet)
    tweet = re.sub(r'bit.ly/\S+','',tweet)
    tweet = tweet.strip('[link]')
    return tweet

In [20]:
df['reviews']=df['reviews'].apply(lambda x: remove_links(x))

In [21]:
df.head()

,reviews,cat
0,RT @TommySobiesski: you could give GameStop the cure to cancer &amp; they’d offer you $3.89,no
1,"This last year has been the hardest of my life - to all the cancer researchers, doctors, nurses and caregivers who are so selflessly dedicated to finding a cure and healing patients, you are my he...",no
2,The year is 2043. You opened your newspaper and read the headlines- - World Health Organization releases a cure for Cancer. - Global warming threat- eliminated! - Enrile celebrating his birthday...,no
3,RT @gorskon: Here we go again. Yet another dubious cancer cure video. Watch to the end and see!,no
4,So sick of festival kids fingering themselves with amethyst and wearing 10 pashimas thinking they're hippies. Real hippies would get together and cure cancer instead of spending $600 to stand arou...,no


In [22]:
def remove_users(tweet):
    tweet = re.sub('(RT\s@[A-Za-z]+[A-Za-z0-9-_]+)','',tweet)
    tweet = re.sub('(@[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet) 
    tweet = re.sub('(#[A-Za-z]+[A-Za-z0-9-_]+)', '', tweet)
    return tweet

In [23]:
df['reviews']=df['reviews'].apply(lambda x: remove_users(x))

In [24]:
df.head()

,reviews,cat
0,: you could give GameStop the cure to cancer &amp; they’d offer you $3.89,no
1,"This last year has been the hardest of my life - to all the cancer researchers, doctors, nurses and caregivers who are so selflessly dedicated to finding a cure and healing patients, you are my he...",no
2,The year is 2043. You opened your newspaper and read the headlines- - World Health Organization releases a cure for Cancer. - Global warming threat- eliminated! - Enrile celebrating his birthday...,no
3,: Here we go again. Yet another dubious cancer cure video. Watch to the end and see!,no
4,So sick of festival kids fingering themselves with amethyst and wearing 10 pashimas thinking they're hippies. Real hippies would get together and cure cancer instead of spending $600 to stand arou...,no


In [25]:
def remove_punc(text):
    no_punc = ''.join([c for c in text if c not in string.punctuation])
    return no_punc

In [26]:
df['reviews'] = df['reviews'].apply(lambda x: remove_punc(x))

In [27]:
df.head()

,reviews,cat
0,you could give GameStop the cure to cancer amp they’d offer you 389,no
1,This last year has been the hardest of my life to all the cancer researchers doctors nurses and caregivers who are so selflessly dedicated to finding a cure and healing patients you are my heroes...,no
2,The year is 2043 You opened your newspaper and read the headlines World Health Organization releases a cure for Cancer Global warming threat eliminated Enrile celebrating his birthday Pengui...,no
3,Here we go again Yet another dubious cancer cure video Watch to the end and see,no
4,So sick of festival kids fingering themselves with amethyst and wearing 10 pashimas thinking theyre hippies Real hippies would get together and cure cancer instead of spending 600 to stand around ...,no


In [28]:
from sklearn.model_selection import train_test_split

In [29]:
X = df['reviews']
y = df['cat']

In [30]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer

In [32]:
text_cal = Pipeline([('tfid', TfidfVectorizer()), ('cal', LinearSVC())])

In [33]:
text_cal.fit(X_train, y_train)

Pipeline(steps=[('tfid', TfidfVectorizer()), ('cal', LinearSVC())])

In [34]:
prediction = text_cal.predict(X_test)

In [35]:
from sklearn import metrics

In [36]:
print(metrics.classification_report(y_test, prediction))

              precision    recall  f1-score   support

          no       0.91      0.99      0.95       390
         yes       0.86      0.40      0.55        62

    accuracy                           0.91       452
   macro avg       0.89      0.70      0.75       452
weighted avg       0.91      0.91      0.89       452



In [37]:
print(metrics.confusion_matrix(y_test, prediction))

[[386   4]
 [ 37  25]]


In [38]:
print(metrics.accuracy_score(y_test, prediction))

0.9092920353982301


In [39]:
import joblib

In [40]:
joblib.dump(text_cal, "cancer_text_classification.pkl")

['cancer_text_classification.pkl']